# GNSS Doppler 기반 GPS 스푸핑 탐지 연구 — 통합 실행 Notebook

## 연구 목적
정상 GPS L1 C/A IQ 생성부터 GNSS-SDR 수신기 처리, 스푸핑 비교, 특징 데이터셋과 탐지 모델 평가까지 **한 Notebook에서 위에서 아래로** 진행합니다.

## 전체 연구 순서

```text
0. 환경·설정 확인
1. generate_iq.py로 정상 IQ 생성
2. manifest·raw IQ 기초 검증
3. 실제 RINEX/SP3 기반 가시 PRN·Doppler truth
4. GNSS-SDR acquisition·tracking·observables
5. 정상 수신 결과와 truth 비교
6. 스푸핑 시나리오 생성·정상/공격 비교
7. 특징 데이터셋·통계/ML/DL baseline
8. TEXBAT/OAKBAT/FGI 외부 일반화
```

각 단계는 이전 단계의 판정을 통과한 뒤 진행합니다.

## 0. 환경과 실험 설정
Notebook 맨 앞에서 정상 IQ 생성 조건을 직접 설정합니다. 현재 generator는 정적 위치를 지원하며, 향후 이동 궤적은 `POSITION_MODE="trajectory"`와 `TRAJECTORY_FILE`을 사용하도록 확장합니다.


In [ ]:
from pathlib import Path
import sys, json, subprocess, tempfile
import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("저장소 루트 또는 notebooks/에서 실행하세요.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
ARTIFACTS = PROJECT_ROOT / "artifacts"

# ── 사용자가 바꾸는 정상 IQ 실험 설정 ──────────────────────────────
RUN_GENERATION = False             # True: 새 IQ 생성, False: 기존 최신 IQ 사용
RUN_RECEIVER = False               # True: 최신 IQ를 GNSS-SDR로 처리
GNSS_SDR_PATH = Path("/usr/bin/gnss-sdr")
GNSS_SDR_CHANNEL_COUNT = 11
SCENARIO_NAME = "seoul-normal-static-2022"
SCENARIO_UTC = "2022-01-01T00:00:00Z"
DURATION_SECONDS = 1
RF_SAMPLE_RATE_HZ = 2_600_000

POSITION_MODE = "static"           # 현재 지원: static / 향후: trajectory
LATITUDE_DEG = 37.5665
LONGITUDE_DEG = 126.9780
ALTITUDE_M = 38.0
TRAJECTORY_FILE = None              # 향후 이동 경로 CSV 경로


ACQ_PRN = "G05"                    # acquisition surface를 볼 PRN
ACQ_COHERENT_MS = 1                 # 1 ms coherent integration
ACQ_NONCOHERENT_MS = 10             # 여러 1 ms 결과를 누적해 피크를 더 선명하게 표시
ACQ_DOPPLER_MIN_HZ = -10_000
ACQ_DOPPLER_MAX_HZ = 10_000
ACQ_DOPPLER_STEP_HZ = 250
ACQ_SURFACE_OUTPUT = None           # None이면 artifacts/figures/acquisition/<run_id>/ 아래 저장

PEAK_RECEIVER_RUN = None            # None: 현재 진행 상황의 최신 receiver run 사용
PEAK_PRN = None                     # None: 사용 가능한 PRN 중 첫 번째 사용
PEAK_MAX_EPOCHS = 1500              # 너무 긴 run은 decimation과 함께 제한
PEAK_EPOCH_STEP = 10                # 1이면 모든 tracking epoch 사용
PEAK_EPOCH_INDEX = None             # None이면 중간 epoch profile 사용
PEAK_DASHBOARD_OUTPUT = None        # 저장할 PNG 경로. None이면 임시 파일 사용

RINEX_NAV_PATH = PROJECT_ROOT / ".tools" / "gps-sdr-sim-src" / "brdc0010.22n"
GPS_SDR_SIM_PATH = PROJECT_ROOT / ".tools" / "gps-sdr-sim-src" / "gps-sdr-sim"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("생성 실행:", RUN_GENERATION)
print("시나리오:", SCENARIO_NAME, SCENARIO_UTC)
print("위치 모드:", POSITION_MODE, (LATITUDE_DEG, LONGITUDE_DEG, ALTITUDE_M))
print("RINEX NAV:", RINEX_NAV_PATH)
print("peak receiver override:", PEAK_RECEIVER_RUN, "peak PRN:", PEAK_PRN)


## 진행 상황
산출물을 기준으로 현재 완료된 단계를 표시합니다. 주요 단계가 끝날 때 이 셀을 다시 실행합니다.

In [ ]:
from gnss_doppler_lab.research_sequence import sequence_status

def show_progress():
    status = sequence_status(ARTIFACTS)
    rows = [
        ("1 정상 IQ 생성", status["01_normal_iq"]["ready"], status["01_normal_iq"]["path"]),
        ("4 GNSS-SDR 관측값", status["02_receiver_processing"]["ready"], status["02_receiver_processing"]["path"]),
        ("6 스푸핑 IQ", status["03_spoofing_comparison"]["ready"], status["03_spoofing_comparison"]["path"]),
        ("6 정상/공격 비교표", status["03_comparison_table"]["ready"], status["03_comparison_table"]["path"]),
        ("7 특징 데이터셋", status["04_detection_dataset"]["ready"], status["04_detection_dataset"]["path"]),
    ]
    print(f"{'연구 단계':<24} {'완료':<6} 산출물")
    print("-" * 100)
    for name, ready, path in rows:
        print(f"{name:<24} {'✅' if ready else '⬜':<6} {path or '-'}")
    return status

status = show_progress()

## 1. 정상 IQ 생성 — `generate_iq.py`

### 입력
`CONFIG`의 UTC·위치·duration·sample rate와 RINEX NAV 설정을 사용합니다.

기존 IQ를 분석할 때는 `False`, 새 run을 만들 때만 `True`로 바꿉니다. Notebook은 실행을 지휘하고 실제 생성은 테스트된 `scripts/generate_iq.py`가 담당합니다.

In [ ]:
if RUN_GENERATION:
    if POSITION_MODE != "static":
        raise NotImplementedError("이동 trajectory IQ 생성은 다음 단계에서 지원합니다. 현재 POSITION_MODE='static'만 가능합니다.")
    runtime_config = {
        "version": 1,
        "scenario": {
            "name": SCENARIO_NAME, "constellation": "GPS", "signal": "L1CA",
            "utc": SCENARIO_UTC, "duration_seconds": DURATION_SECONDS,
            "position": {"type": "static", "latitude_deg": LATITUDE_DEG,
                         "longitude_deg": LONGITUDE_DEG, "altitude_m": ALTITUDE_M},
        },
        "input": {"rinex_nav": str(RINEX_NAV_PATH.resolve())},
        "output": {"root": str((ARTIFACTS / "rf_runs").resolve()),
                   "rf_sample_rate_hz": RF_SAMPLE_RATE_HZ, "sample_format": "s8_iq"},
        "simulator": {"executable": str(GPS_SDR_SIM_PATH.resolve())},
    }
    print(yaml.safe_dump(runtime_config, sort_keys=False, allow_unicode=True))
    with tempfile.TemporaryDirectory(prefix="gnss-iq-") as temp_dir:
        config_path = Path(temp_dir) / "iq_config.yaml"
        config_path.write_text(yaml.safe_dump(runtime_config, sort_keys=False), encoding="utf-8")
        command = [sys.executable, str(PROJECT_ROOT / "scripts" / "generate_iq.py"), "generate", str(config_path),
                   "--executable", str(GPS_SDR_SIM_PATH.resolve())]
        print("실행:", " ".join(command))
        result = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"IQ 생성 실패: exit={result.returncode}")
    print("정상 IQ 생성 완료")
else:
    print("생성 건너뜀: 기존 최신 정상 IQ를 사용합니다. 새로 생성하려면 맨 앞 설정에서 RUN_GENERATION=True")


## 2. 정상 IQ 기초 검증

### 중간 데이터
최신 완전 run의 manifest, 표본 수, duration, RMS, peak, DC offset을 확인합니다.

In [ ]:
import numpy as np
from pprint import pprint
from gnss_doppler_lab.research_sequence import latest_run, load_run_manifest
from gnss_doppler_lab.iq_visualization import load_s8_iq, summarize_iq

run_dir = latest_run(ARTIFACTS / "rf_runs")
manifest = load_run_manifest(run_dir)
iq_path = run_dir / "gps_l1ca_s8_iq.bin"
sample_rate_hz = float(manifest.get("iq", {}).get("rf_sample_rate_hz", 2_600_000))
iq = load_s8_iq(iq_path, max_complex_samples=int(sample_rate_hz))
print("RUN:", run_dir.name)
pprint(manifest)
print(json.dumps(summarize_iq(iq, sample_rate_hz=sample_rate_hz), indent=2))
print("첫 10개 IQ:", iq[:10])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
n=min(5000,iq.size); t=np.arange(n)/sample_rate_hz*1e3
ax[0,0].plot(t,iq.real[:n],lw=.6,label='I'); ax[0,0].plot(t,iq.imag[:n],lw=.6,label='Q')
ax[0,0].set(title='Time-domain I/Q',xlabel='Time (ms)',ylabel='ADC'); ax[0,0].legend(); ax[0,0].grid(alpha=.25)
pts=iq[::max(1,iq.size//40000)][:40000]
ax[0,1].hexbin(pts.real,pts.imag,gridsize=75,mincnt=1,bins='log'); ax[0,1].set(title='I/Q density',xlabel='I',ylabel='Q'); ax[0,1].set_aspect('equal')
nfft=min(65536,2**int(np.floor(np.log2(iq.size)))); x=iq[:nfft]*np.hanning(nfft)
p=np.fft.fftshift(np.fft.fft(x)); db=20*np.log10(np.maximum(abs(p),1e-12)); db-=db.max()
f=np.fft.fftshift(np.fft.fftfreq(nfft,1/sample_rate_hz))/1e6
ax[1,0].plot(f,db,lw=.7); ax[1,0].set_ylim(-100,5); ax[1,0].set(title='Relative spectrum',xlabel='Offset (MHz)',ylabel='dB'); ax[1,0].grid(alpha=.25)
count=min(iq.size,int(sample_rate_hz*.2)); ax[1,1].specgram(iq[:count],NFFT=2048,Fs=sample_rate_hz,noverlap=1536,cmap='magma'); ax[1,1].set(title='Spectrogram',xlabel='Time (s)',ylabel='Hz')
plt.show()

### 2단계 판정
- [ ] duration·sample 수가 설정과 일치
- [ ] clipping과 큰 DC offset 없음
- [ ] I/Q 분포가 심하게 한 축으로 치우치지 않음
- [ ] 비정상 협대역 tone·aliasing 없음
- [ ] manifest에 UTC·위치·NAV/IQ hash 기록

그래프는 기초 건전성 검사이며 PRN 정확성은 다음 단계에서 검증합니다.

## 3. 가시 위성 및 Doppler truth

### 입력
실제 RINEX NAV/SP3, 시나리오 UTC·수신기 위치·trajectory입니다.

### 중간 확인
PRN별 satellite position/velocity, azimuth, elevation, range rate, 예상 L1 Doppler를 표와 곡선으로 확인합니다.

In [ ]:
truth_candidates = sorted((ARTIFACTS / "truth").glob("*/doppler_truth.csv")) if (ARTIFACTS / "truth").exists() else []
if not truth_candidates:
    print("Doppler truth가 아직 없습니다. 다음 구현 목표: RINEX/UTC/위치 → PRN별 az/el/range-rate/doppler_hz")
else:
    print("최신 truth:", truth_candidates[-1])

### 3단계 판정
- [ ] elevation mask 이상의 전체 가시 PRN 포함
- [ ] Doppler 부호와 단위 검증
- [ ] 한 PRN 고정 검사가 아닌 전체 geometry invariant 검증
- [ ] 사용 RINEX/SP3와 계산 코드 버전 보존

## 4. GNSS-SDR acquisition·tracking·observables
정상 IQ를 GNSS-SDR에 입력해 PRN, acquisition Doppler, tracking lock, C/N₀, correlator, observables를 추출합니다.

In [ ]:
import csv
from gnss_doppler_lab.gnss_sdr import run_receiver

latest_rf_run = latest_run(ARTIFACTS / "rf_runs")
receiver_target = ARTIFACTS / "receiver_runs" / latest_rf_run.name
if RUN_RECEIVER:
    if receiver_target.exists():
        print("기존 receiver run 재사용:", receiver_target)
    else:
        receiver_manifest = run_receiver(
            latest_rf_run / "manifest.json",
            ARTIFACTS / "receiver_runs",
            executable=GNSS_SDR_PATH,
            channel_count=GNSS_SDR_CHANNEL_COUNT,
        )
        print("GNSS-SDR 처리 완료:", receiver_manifest)
else:
    print("수신기 실행 건너뜀. 새 RF run을 처리하려면 RUN_RECEIVER=True")

status = show_progress()
receiver_dir = Path(status["02_receiver_processing"]["path"]) if status["02_receiver_processing"]["ready"] else None
tracking_path = receiver_dir / "tracking.csv" if receiver_dir else None
summary_path = receiver_dir / "tracking_summary.csv" if receiver_dir else None
rows = list(csv.DictReader(tracking_path.open())) if tracking_path else []
summaries = list(csv.DictReader(summary_path.open())) if summary_path else []
if not rows:
    print("GNSS-SDR 결과가 아직 없습니다. RUN_RECEIVER=True로 실행하세요.")
else:
    print("추적 행:", len(rows), "PRN:", sorted({r['prn'] for r in rows}))
    print(f"{'PRN':<5} {'epochs':>7} {'median Doppler (Hz)':>20} {'median C/N0 (dB-Hz)':>22}")
    for row in sorted(summaries, key=lambda x: x['prn']):
        print(f"{row['prn']:<5} {int(row['epoch_count']):>7} {float(row['median_doppler_hz']):>20.2f} {float(row['median_cn0_db_hz']):>22.2f}")
    fig, ax = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
    for prn in sorted({r['prn'] for r in rows}):
        group = [r for r in rows if r['prn'] == prn]
        time = [float(r['time_s']) for r in group]
        ax[0].plot(time, [float(r['carrier_doppler_hz']) for r in group], label=prn)
        ax[1].plot(time, [float(r['CN0_SNV_dB_Hz']) for r in group], label=prn)
    ax[0].set(title='GNSS-SDR carrier Doppler', xlabel='Time (s)', ylabel='Doppler (Hz)')
    ax[1].set(title='GNSS-SDR C/N0', xlabel='Time (s)', ylabel='C/N0 (dB-Hz)')
    for axis in ax: axis.grid(alpha=.25)
    ax[0].legend(ncol=3); plt.show()


## 4A. acquisition delay-Doppler surface
수신기 tracking 전에, 정상 IQ에서 선택한 PRN의 **delay × Doppler correlation surface**를 직접 계산해 acquisition 단계의 3D peak를 확인합니다. 이 그래프가 우리가 말한 “피크가 3D 산처럼 보인다”는 개념에 가장 가깝습니다.

- 입력: `artifacts/rf_runs/<run_id>/gps_l1ca_s8_iq.bin`
- X축: code delay chips
- Y축: Doppler Hz
- Z축/색: correlation magnitude
- 출력: `artifacts/figures/acquisition/<run_id>/`


In [ ]:
from IPython.display import Image, display
from gnss_doppler_lab.acquisition_surface import (
    read_s8_iq,
    compute_acquisition_surface,
    render_acquisition_surface,
    save_surface_summary,
)

latest_rf_run = latest_run(ARTIFACTS / "rf_runs")
rf_manifest_path = latest_rf_run / "manifest.json"
if not rf_manifest_path.exists():
    print("acquisition surface 대기: RF run manifest가 필요합니다.")
else:
    rf_manifest = json.loads(rf_manifest_path.read_text())
    sample_rate_hz = rf_manifest["iq"]["rf_sample_rate_hz"]
    iq_path = latest_rf_run / rf_manifest["iq"]["path"]
    needed_samples = int(round(sample_rate_hz * 0.001 * ACQ_NONCOHERENT_MS))
    iq = read_s8_iq(iq_path, samples=needed_samples)
    acq_surface = compute_acquisition_surface(
        iq,
        ACQ_PRN,
        sample_rate_hz,
        coherent_ms=ACQ_COHERENT_MS,
        noncoherent_ms=ACQ_NONCOHERENT_MS,
        doppler_min_hz=ACQ_DOPPLER_MIN_HZ,
        doppler_max_hz=ACQ_DOPPLER_MAX_HZ,
        doppler_step_hz=ACQ_DOPPLER_STEP_HZ,
    )
    acq_dir = ARTIFACTS / "figures" / "acquisition" / latest_rf_run.name
    if ACQ_SURFACE_OUTPUT:
        acq_png = Path(ACQ_SURFACE_OUTPUT)
    else:
        acq_png = acq_dir / f"{acq_surface.prn}_acquisition_delay_doppler_surface_{ACQ_NONCOHERENT_MS}ms.png"
    render_acquisition_surface(
        acq_surface,
        acq_png,
        title=f"{latest_rf_run.name} | {acq_surface.prn} acquisition correlation surface",
    )
    acq_json = acq_png.with_suffix(".json")
    save_surface_summary(acq_surface, acq_json, source_iq=iq_path)
    print("acquisition surface:", acq_png)
    print("summary:", acq_json)
    print(
        "peak:",
        f"delay={acq_surface.peak_code_delay_chips:.1f} chips,",
        f"doppler={acq_surface.peak_doppler_hz:.0f} Hz,",
        f"peak/2nd={acq_surface.peak_to_second_ratio:.2f}",
    )
    display(Image(filename=str(acq_png)))


## 4B. tracking correlator peak
GNSS-SDR tracking dump(`raw/epl_tracking_ch_*.mat`)에서 **PRN별 correlator tap magnitude**를 읽어 실제 수신 신호의 peak slice를 확인합니다. 현재 단계는 tracking dump가 제공하는 tapped-delay profile(E/P/L 등) 기반이며, raw IQ에서의 full delay-Doppler requery는 후속 확장입니다.


In [ ]:
from IPython.display import Image, display
from gnss_doppler_lab.tracking_peaks import (
    available_tracking_prns,
    load_receiver_tracking_peak_series,
    render_tracking_peak_dashboard,
)

if receiver_dir is None or not receiver_dir.exists():
    print("tracking correlator peak 대기: 4단계 receiver run이 필요합니다.")
else:
    peak_receiver_dir = Path(PEAK_RECEIVER_RUN) if PEAK_RECEIVER_RUN else receiver_dir
    peak_prns = available_tracking_prns(peak_receiver_dir)
    print("peak receiver run:", peak_receiver_dir)
    print("available PRNs:", peak_prns)
    selected_prn = PEAK_PRN or (peak_prns[0] if peak_prns else None)
    if selected_prn is None:
        print("분석 가능한 PRN이 없습니다.")
    else:
        peak_series = load_receiver_tracking_peak_series(
            peak_receiver_dir,
            selected_prn,
            max_epochs=PEAK_MAX_EPOCHS,
            epoch_step=PEAK_EPOCH_STEP,
        )
        if PEAK_DASHBOARD_OUTPUT:
            peak_output = Path(PEAK_DASHBOARD_OUTPUT)
        else:
            peak_output = Path(tempfile.NamedTemporaryFile(suffix=".png", delete=False).name)
        peak_dashboard = render_tracking_peak_dashboard(
            peak_series,
            output_path=peak_output,
            title=f"{peak_receiver_dir.name} | {peak_series.prn} tracking correlator peak",
            epoch_index=PEAK_EPOCH_INDEX,
        )
        print("selected PRN:", peak_series.prn, "channel:", peak_series.channel)
        print("tap names:", peak_series.tap_names, "epochs:", len(peak_series.time_s))
        print("source MAT:", peak_series.source_mat_path.name)
        print("dashboard:", peak_dashboard)
        display(Image(filename=str(peak_dashboard)))


### 4단계 판정
- [ ] 예상 가시 PRN acquisition
- [ ] tracking lock·C/N₀ 유지
- [ ] 측정 Doppler와 Doppler truth의 PRN·부호·크기 일치
- [ ] observables와 GNSS-SDR 설정 보존

## 5. 정상 수신 결과 최종 검증
PRN별 `measured Doppler - truth Doppler` residual, acquisition 성공률, tracking 연속성, 가능하면 PVT 오차를 확인합니다.

In [ ]:
if not rows or not truth_candidates:
    print("정상 수신 검증 대기: 3단계 truth와 4단계 GNSS-SDR observables가 모두 필요합니다.")
else:
    print("다음 구현에서 시간/PRN 기준으로 truth와 observables를 정렬해 residual을 계산합니다.")

## 6. 스푸핑 시나리오 생성과 정상/공격 비교
정상 수신 체인을 먼저 통과한 뒤 스푸핑 시나리오를 생성합니다. 공격 시작시각, 유형, PRN 범위, power advantage를 manifest에 기록합니다.

In [ ]:
status = show_progress()
spoof_dir = Path(status["03_spoofing_comparison"]["path"]) if status["03_spoofing_comparison"]["ready"] else None
if spoof_dir is None:
    print("스푸핑 시나리오가 아직 없습니다. 정상 체인 검증 후 생성합니다.")
else:
    print("스푸핑 run:", spoof_dir)
    print("다음으로 정상/공격의 Doppler, C/N0, correlator, clock, PVT를 같은 시간축에서 비교합니다.")

### 6단계 판정
- [ ] 정상/공격의 UTC·위치·trajectory·sample rate 통제
- [ ] 공격 metadata와 시작 시각 명시
- [ ] power나 파일명만으로 label이 노출되지 않음
- [ ] 공격 전후 receiver internal 변화 확인

## 7. 특징 데이터셋과 통계·ML·DL baseline
수신기 관측값을 시간 window로 만들고 random row split이 아닌 run/scenario 단위로 분리합니다.

In [ ]:
status = show_progress()
dataset_path = Path(status["04_detection_dataset"]["path"]) if status["04_detection_dataset"]["ready"] else None
if dataset_path is None:
    print("특징 데이터셋이 아직 없습니다. 4~6단계 검증 후 생성합니다.")
else:
    print("데이터셋:", dataset_path)
    print("평가 순서: 물리 기반 통계 → classical ML → 시계열 DL")

### 7단계 판정
- [ ] run/scenario 단위 train/validation/test 분리
- [ ] label leakage 검사
- [ ] 통계 baseline보다 복잡한 모델의 이점 검증
- [ ] PR-AUC, F1, false alarm, detection delay 보고

## 8. TEXBAT/OAKBAT/FGI 외부 일반화
내부 시뮬레이션으로 모델을 고정한 뒤 공개 raw RF 데이터를 같은 수신기/특징 schema로 재처리합니다.

In [ ]:
external_root = ARTIFACTS / "external_validation"
print("외부 검증 데이터 준비:", external_root.exists())
print("목표: TEXBAT/OAKBAT/FGI-JSDR → GNSS-SDR/동일 feature → source-holdout 평가")

## 최종 판정

연구 결과로 주장하기 위한 최소 조건:

- [ ] 실제 ephemeris 기반 정상 Doppler truth가 검증됨
- [ ] 생성 IQ가 GNSS-SDR acquisition/tracking/PVT를 통과함
- [ ] 정상/스푸핑 비교에서 물리적으로 설명 가능한 차이가 있음
- [ ] 데이터 분리와 label leakage 통제가 확인됨
- [ ] 통계·ML·DL을 동일 split에서 비교함
- [ ] 외부 RF 데이터에서 일반화를 평가함
- [ ] config·manifest·hash·코드 버전으로 실험을 재현할 수 있음

이 Notebook은 연구 진행 화면이며, 반복 실행 로직은 `scripts/`, 계산 로직은 `src/`, 원자료와 실험 산출물은 `artifacts/`에 유지합니다.